# 06 · Flagging students before the exam

**Target:** bottom quartile of the student's own cohort (so every semester contributes its weakest 25%).
**Validation:** leave one semester out: train on 7 semesters, predict the unseen 8th, as an instructor would use it.
**Models:** regularized logistic regression (interpretable baseline) vs histogram gradient boosting.
Every run is logged to MLflow (`make mlflow` opens the UI).

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

In [2]:
from so_ell import models, pipeline
features = pd.read_parquet(config.PROCESSED_DIR / "student_exam_features.parquet")
print("features used:", models.PRACTICE_FEATURES)

features used: ['parsons__mean_first_score', 'mchoice__mean_first_score', 'activecode__mean_best_score', 'all_activity__total_events', 'all_activity__active_days', 'coverage__unique_chapters', 'coverage__chapter_diversity', 'timing__share_last_7d', 'consistency__problems_per_active_day', 'mix__coding_share']


## 1. Model definitions and cross-validation

In [3]:
show(models.make_model, models.add_target, models.precision_at_top, models.leave_one_semester_out)

`so_ell.models.make_model`
```python
def make_model(kind: str) -> Pipeline:
    if kind == "logistic":
        return Pipeline(
            [
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000, C=1.0)),
            ]
        )
    if kind == "gradient_boosting":
        return Pipeline(
            [
                (
                    "clf",
                    HistGradientBoostingClassifier(
                        max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0,
                        early_stopping=False, random_state=config.RANDOM_SEED,
                    ),
                )
            ]
        )
    raise ValueError(kind)
```

`so_ell.models.add_target`
```python
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    out = add_cohort(df)
    out["score_rank"] = out.groupby("cohort")["score_pct"].rank(pct=True)
    out["at_risk"] = (out["score_rank"] <= 0.25).astype(int)
    return out
```

`so_ell.models.precision_at_top`
```python
def precision_at_top(y: np.ndarray, proba: np.ndarray, share: float = 0.10) -> float:
    k = max(1, int(round(len(y) * share)))
    top = np.argsort(-proba)[:k]
    return float(y[top].mean())
```

`so_ell.models.leave_one_semester_out`
```python
def leave_one_semester_out(df: pd.DataFrame, features: list[str], kind: str, name: str) -> CVResult:
    use = add_target(df).dropna(subset=["at_risk"]).reset_index(drop=True)
    features = [f for f in features if f in use.columns]
    X, y = use[features], use["at_risk"].to_numpy()
    groups = use["semester_raw"].astype(str).to_numpy()
    oof = np.full(len(use), np.nan)
    rows = []
    for train, test in LeaveOneGroupOut().split(X, y, groups):
        model = make_model(kind).fit(X.iloc[train], y[train])
        proba = model.predict_proba(X.iloc[test])[:, 1]
        oof[test] = proba
        rows.append(
            {
                "model": name,
                "held_out_semester": groups[test][0],
                "n_test": len(test),
                "auc": roc_auc_score(y[test], proba),
                "average_precision": average_precision_score(y[test], proba),
            }
        )
    summary = {
        "auc": roc_auc_score(y, oof),
        "average_precision": average_precision_score(y, oof),
        "brier": brier_score_loss(y, oof),
        "precision_top10pct": precision_at_top(y, oof),
        "base_rate": float(y.mean()),
        "n": len(use),
        "n_features": len(features),
    }
    importance = None
    if kind == "gradient_boosting":
        full = make_model(kind).fit(X, y)
        imp = permutation_importance(full, X, y, scoring="roc_auc", n_repeats=10, random_state=config.RANDOM_SEED)
        importance = pd.DataFrame(
            {"feature": features, "auc_drop_mean": imp.importances_mean, "auc_drop_sd": imp.importances_std}
        ).sort_values("auc_drop_mean", ascending=False)
    elif kind == "logistic":
        full = make_model(kind).fit(X, y)
        importance = pd.DataFrame({"feature": features, "std_coef": full.named_steps["clf"].coef_[0]}).sort_values("std_coef")
    oof_frame = use[["semester_raw", "anon_student_id", "midterm", "at_risk"]].assign(model=name, proba=oof)
    return CVResult(name, pd.DataFrame(rows), summary, oof_frame, importance)
```

## 2. Run the suite
The last three rows are midterm 2 only: does practice add anything once midterm 1 is known?

In [4]:
show(models.run_model_suite)
results = models.run_model_suite(features)
comparison = pd.DataFrame([{"model": k} | r.summary for k, r in results.items()])
comparison.to_csv(T / "at_risk_model_comparison.csv", index=False)
pd.concat([r.folds for r in results.values()], ignore_index=True).to_csv(T / "at_risk_folds.csv", index=False)
results["gradient_boosting"].importance.to_csv(T / "at_risk_importance_gbm.csv", index=False)
results["logistic"].importance.to_csv(T / "at_risk_coefficients_logistic.csv", index=False)
comparison.round(3)

`so_ell.models.run_model_suite`
```python
def run_model_suite(features: pd.DataFrame) -> dict[str, CVResult]:
    """Main comparison: logistic vs gradient boosting, plus a prior-ability baseline."""
    results = {
        "logistic": leave_one_semester_out(features, PRACTICE_FEATURES, "logistic", "logistic"),
        "gradient_boosting": leave_one_semester_out(features, PRACTICE_FEATURES, "gradient_boosting", "gradient_boosting"),
    }

    # Midterm 2 only: does ebook practice add anything once we know the mid1 score?
    mid = add_cohort(features)
    mid1 = mid.loc[mid["midterm"].eq("mid1"), ["semester_raw", "anon_student_id", "score_pct"]].rename(
        columns={"score_pct": "mid1_score"}
    )
    mid2 = mid.loc[mid["midterm"].eq("mid2")].merge(mid1, on=["semester_raw", "anon_student_id"], how="inner")
    results["mid2_mid1_only"] = leave_one_semester_out(mid2, ["mid1_score"], "logistic", "mid2_mid1_only")
    results["mid2_practice_only"] = leave_one_semester_out(mid2, PRACTICE_FEATURES, "logistic", "mid2_practice_only")
    results["mid2_mid1_plus_practice"] = leave_one_semester_out(
        mid2, ["mid1_score"] + PRACTICE_FEATURES, "logistic", "mid2_mid1_plus_practice"
    )
    return results
```

,model,auc,average_precision,brier,precision_top10pct,base_rate,n,n_features
0,logistic,0.780,0.500,0.155,0.573,0.248,2745,10
1,gradient_boosting,0.759,0.450,0.167,0.507,0.248,2745,10
2,mid2_mid1_only,0.824,0.593,0.138,0.711,0.244,1348,1
3,mid2_practice_only,0.752,0.444,0.166,0.511,0.244,1348,10
4,mid2_mid1_plus_practice,0.828,0.587,0.144,0.652,0.244,1348,11


Per held-out semester (AUC):

In [5]:
pd.concat([r.folds for r in results.values()]).pivot(index="held_out_semester", columns="model", values="auc").round(3)

model,gradient_boosting,logistic,mid2_mid1_only,mid2_mid1_plus_practice,mid2_practice_only
held_out_semester,,,,,
F21,0.777,0.803,0.881,0.875,0.782
F22,0.839,0.862,0.877,0.913,0.895
F23,0.797,0.812,0.883,0.902,0.836
F24,0.767,0.788,0.850,0.852,0.745
F25,0.711,0.735,0.845,0.845,0.713
W22,0.772,0.781,0.819,0.776,0.734
W23,0.833,0.849,0.845,0.856,0.842
W24,0.732,0.759,0.862,0.845,0.746


## 3. What the models rely on

In [6]:
results["gradient_boosting"].importance.round(4)

,feature,auc_drop_mean,auc_drop_sd
0,parsons__mean_first_score,0.1825,0.0101
7,timing__share_last_7d,0.0467,0.0021
1,mchoice__mean_first_score,0.0464,0.0040
2,activecode__mean_best_score,0.0370,0.0030
3,all_activity__total_events,0.0370,0.0028
5,coverage__unique_chapters,0.0223,0.0022
8,consistency__problems_per_active_day,0.0220,0.0012
4,all_activity__active_days,0.0140,0.0009
9,mix__coding_share,0.0134,0.0010
6,coverage__chapter_diversity,0.0129,0.0009


In [7]:
results["logistic"].importance.round(3)

,feature,std_coef
0,parsons__mean_first_score,-0.986
1,mchoice__mean_first_score,-0.348
2,activecode__mean_best_score,-0.254
5,coverage__unique_chapters,-0.243
7,timing__share_last_7d,-0.229
3,all_activity__total_events,-0.150
9,mix__coding_share,-0.021
8,consistency__problems_per_active_day,0.024
6,coverage__chapter_diversity,0.042
4,all_activity__active_days,0.101


## 4. Calibration (out of fold)

In [8]:
show(models.calibration_table)
oof = pd.concat([results["logistic"].oof, results["gradient_boosting"].oof], ignore_index=True)
cal = models.calibration_table(oof)
cal.to_csv(T / "at_risk_calibration.csv", index=False)
cal.round(3)

`so_ell.models.calibration_table`
```python
def calibration_table(oof: pd.DataFrame, bins: int = 10) -> pd.DataFrame:
    use = oof.copy()
    use["bin"] = pd.qcut(use["proba"].rank(method="first"), bins, labels=False) + 1
    return (
        use.groupby(["model", "bin"], as_index=False)
        .agg(mean_predicted=("proba", "mean"), observed_rate=("at_risk", "mean"), n=("at_risk", "size"))
    )
```

,model,bin,mean_predicted,observed_rate,n
0,gradient_boosting,1,0.018,0.040,376
1,gradient_boosting,2,0.046,0.039,283
2,gradient_boosting,3,0.079,0.071,238
3,gradient_boosting,4,0.124,0.170,223
4,gradient_boosting,5,0.180,0.251,223
5,gradient_boosting,6,0.253,0.231,234
6,gradient_boosting,7,0.340,0.298,272
7,gradient_boosting,8,0.441,0.388,255
8,gradient_boosting,9,0.556,0.477,300
9,gradient_boosting,10,0.733,0.490,341


## 5. How early does the signal appear?
The same models, trained on features from practice at least N days before the exam (built in notebook 04).

In [9]:
show(models.lead_time_curve)
by_cutoff = {0: features}
for c in [c for c in pipeline.LEAD_TIME_CUTOFFS if c]:
    by_cutoff[c] = pd.read_parquet(config.PROCESSED_DIR / f"student_exam_features_cutoff_{c}d.parquet")
lead = models.lead_time_curve(by_cutoff)
lead.to_csv(T / "at_risk_lead_time.csv", index=False)
lead.pivot(index="cutoff_days", columns="model", values="auc").sort_index(ascending=False).round(3)

`so_ell.models.lead_time_curve`
```python
def lead_time_curve(features_by_cutoff: dict[int, pd.DataFrame]) -> pd.DataFrame:
    """Out-of-fold AUC when the model only sees practice from >= N days before the exam."""
    rows = []
    for cutoff, frame in sorted(features_by_cutoff.items()):
        for kind in ["logistic", "gradient_boosting"]:
            result = leave_one_semester_out(frame, PRACTICE_FEATURES, kind, f"{kind}_cutoff_{cutoff}")
            rows.append({"cutoff_days": cutoff, "model": kind} | result.summary)
    return pd.DataFrame(rows)
```

model,gradient_boosting,logistic
cutoff_days,,
28,0.704,0.696
21,0.719,0.700
14,0.727,0.708
7,0.731,0.722
3,0.730,0.733
0,0.759,0.780


## 6. Log to MLflow

In [10]:
show(models.log_to_mlflow)
models.log_to_mlflow(results, lead)

`so_ell.models.log_to_mlflow`
```python
def log_to_mlflow(results: dict[str, CVResult], lead_time: pd.DataFrame | None = None) -> bool:
    """Log each model as an MLflow run. Returns False when MLflow isn't installed."""
    try:
        import mlflow
    except ImportError:
        return False
    config.MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(f"sqlite:///{(config.MLRUNS_DIR / 'mlflow.db').resolve().as_posix()}")
    mlflow.set_experiment("so-ell-at-risk")
    for name, result in results.items():
        with mlflow.start_run(run_name=name):
            mlflow.log_params({"model": name, "validation": "leave_one_semester_out", "n_features": result.summary["n_features"]})
            mlflow.log_metrics({k: float(v) for k, v in result.summary.items() if k not in {"n", "n_features"}})
            for row in result.folds.itertuples():
                mlflow.log_metric(f"auc_{row.held_out_semester}", float(row.auc))
            if result.importance is not None:
                mlflow.log_text(result.importance.to_csv(index=False), "importance.csv")
    if lead_time is not None:
        with mlflow.start_run(run_name="lead_time_curve"):
            for row in lead_time.itertuples():
                mlflow.log_metric(f"auc_{row.model}", float(row.auc), step=int(row.cutoff_days))
            mlflow.log_text(lead_time.to_csv(index=False), "lead_time.csv")
    return True
```

2026/10/18 20:56:21 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/claude/so-ell/project/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/10/18 20:56:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/18 20:56:21 INFO mlflow.store.db.utils: Updating database tables


2026/10/18 20:56:22 INFO mlflow.tracking.fluent: Experiment with name 'so-ell-at-risk' does not exist. Creating a new experiment.


True